<a href="https://colab.research.google.com/github/Heman659-crypto/Used_-Car_data_preprocessing-/blob/main/Day12_Used_Car_Data_Preprocessing_Complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Used Car Data Preprocessing

### Introduction
This notebook performs a complete preprocessing workflow for the Used Car Resale Dataset, including inspection, outlier handling, categorical encoding, scaling, train-test splitting, and data-leakage prevention.

## Objective
1. Inspect the dataset.
2. Identify features and target.
3. Check missing values and duplicates.
4. Handle outliers using IQR.
5. Encode categorical variables.
6. Scale numerical features.
7. Split train/test data.
8. Fit transformations only on training data.
9. Verify and save processed data.

## Dataset Description
The dataset contains used-car information including brand, year, mileage, engine capacity, power, fuel type, transmission, city, seller type, condition, ownership history, accidents, service score, and resale price.

**Target:** `Resale_Price_Lakh`.

## Step 1: Import Required Libraries
Pandas and NumPy are used for data manipulation, while Scikit-learn is used for splitting and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
pd.set_option("display.max_columns", None)

## Step 2: Upload the Dataset
Upload the CSV file from your local device into Google Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Day12_Used_Car_Preprocessing_Dataset.csv to Day12_Used_Car_Preprocessing_Dataset (2).csv


## Step 3: Load the Dataset
The uploaded CSV is loaded into a Pandas DataFrame and the first rows are displayed.

In [ ]:
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
df.head()

,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


## Step 4: Initial Dataset Inspection
Check shape, column names, data types, and general dataset information.

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nDataset Information:")
df.info()

Dataset Shape: (320, 15)

Column Names: ['Car_ID', 'Brand', 'Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type', 'Condition', 'Previous_Owners', 'Accidents_Reported', 'Service_Score', 'Resale_Price_Lakh']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-

## Step 5: Check Missing Values and Duplicate Rows
Missing values and duplicate records are checked before preprocessing.

In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

Missing Values:
 Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

Duplicate Rows: 0


## Step 6: Statistical Summary
Descriptive statistics help understand numerical distributions and quartiles needed for IQR outlier detection.

In [ ]:
df.describe()

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


## Step 7: Separate Features and Target Variable
`Resale_Price_Lakh` is the target. `Car_ID` is removed because it is only an identifier.

In [ ]:
target = "Resale_Price_Lakh"
X = df.drop(columns=[target, "Car_ID"])
y = df[target]
print("Features:", X.shape)
print("Target:", y.shape)

Features: (320, 13)
Target: (320,)


## Step 8: Split Data into Training and Testing Sets
The data is split into 80% training and 20% testing **before preprocessing** to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (256, 13)
Testing: (64, 13)


# Outlier Detection and Handling
## Step 9: Identify Numerical Features
IQR-based outlier detection will be applied to the numerical variables.

In [ ]:
numeric_features = ["Year","Mileage_Km","Engine_CC","Power_BHP","Previous_Owners","Accidents_Reported","Service_Score"]
print(numeric_features)

['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners', 'Accidents_Reported', 'Service_Score']


## Step 10: Detect Outliers Using the IQR Method
**IQR = Q3 − Q1**

**Lower Bound = Q1 − 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

Bounds are calculated from training data only.

In [ ]:
outlier_data = []
for col in numeric_features:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((X_train[col] < lower_bound) | (X_train[col] > upper_bound)).sum()
    outlier_data.append([col,Q1,Q3,IQR,lower_bound,upper_bound,outliers])
outlier_report = pd.DataFrame(outlier_data, columns=["Feature","Q1","Q3","IQR","Lower Bound","Upper Bound","Number of Outliers"])
outlier_report

,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Number of Outliers
0,Year,2017.000,2022.00,5.000,2009.5000,2029.5000,0
1,Mileage_Km,46076.750,97695.75,51619.000,-31351.7500,175124.2500,2
2,Engine_CC,1013.250,1644.75,631.500,66.0000,2592.0000,6
3,Power_BHP,129.675,171.15,41.475,67.4625,233.3625,5
4,Previous_Owners,1.000,2.00,1.000,-0.5000,3.5000,10
5,Accidents_Reported,0.000,0.00,0.000,0.0000,0.0000,47
6,Service_Score,65.750,86.00,20.250,35.3750,116.3750,0


## Step 11: Handle Outliers
Extreme values are capped at IQR boundaries instead of deleting rows. The boundaries are learned during `fit()` on training data.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, multiplier=1.5): self.multiplier = multiplier
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        self.lower_ = self.q1_ - self.multiplier * self.iqr_
        self.upper_ = self.q3_ + self.multiplier * self.iqr_
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# Categorical Data Encoding
## Step 12: Identify Nominal and Ordinal Features
Nominal features have no natural order and use One-Hot Encoding. `Condition` has a natural order and uses Ordinal Encoding.

In [ ]:
nominal_features = ["Brand","Fuel_Type","Transmission","City","Seller_Type"]
ordinal_features = ["Condition"]
print("Nominal:", nominal_features)
print("Ordinal:", ordinal_features)

Nominal: ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
Ordinal: ['Condition']


## Step 13: Define the Order for Condition
The order is: **Poor → Fair → Good → Excellent**.

In [ ]:
condition_order = [["Poor","Fair","Good","Excellent"]]

# Feature Scaling and Preprocessing Pipeline
## Step 14: Create the Numerical Pipeline
Numerical data undergoes IQR outlier capping followed by `StandardScaler`.

In [ ]:
numeric_pipeline = Pipeline([("outlier_handling", IQRClipper(1.5)), ("scaling", StandardScaler())])

## Step 15: Create the Ordinal Encoding Pipeline
The ordered `Condition` categories are encoded numerically.

In [ ]:
ordinal_pipeline = Pipeline([("ordinal_encoding", OrdinalEncoder(categories=condition_order, handle_unknown="use_encoded_value", unknown_value=-1))])

## Step 16: Create the Nominal Encoding Pipeline
Nominal categorical variables are converted into binary columns using One-Hot Encoding.

In [ ]:
nominal_pipeline = Pipeline([("one_hot_encoding", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

## Step 17: Combine All Preprocessing Steps
A `ColumnTransformer` applies the correct preprocessing pipeline to each group of columns.

In [ ]:
preprocessor = ColumnTransformer([("numerical", numeric_pipeline, numeric_features),("ordinal", ordinal_pipeline, ordinal_features),("nominal", nominal_pipeline, nominal_features)])

# Apply Preprocessing Without Data Leakage
## Step 18: Fit the Preprocessor on Training Data
Use `fit_transform()` only on training data and `transform()` on testing data. This prevents information from the test set influencing preprocessing.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Training and testing data processed successfully.")

Training and testing data processed successfully.


## Step 19: Retrieve Processed Feature Names
Feature names are extracted after encoding.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 37
['numerical__Year' 'numerical__Mileage_Km' 'numerical__Engine_CC'
 'numerical__Power_BHP' 'numerical__Previous_Owners'
 'numerical__Accidents_Reported' 'numerical__Service_Score'
 'ordinal__Condition' 'nominal__Brand_Honda' 'nominal__Brand_Hyundai'
 'nominal__Brand_Kia' 'nominal__Brand_Mahindra' 'nominal__Brand_Maruti'
 'nominal__Brand_Renault' 'nominal__Brand_Skoda' 'nominal__Brand_Tata'
 'nominal__Brand_Toyota' 'nominal__Brand_Volkswagen'
 'nominal__Fuel_Type_CNG' 'nominal__Fuel_Type_Diesel'
 'nominal__Fuel_Type_Electric' 'nominal__Fuel_Type_Petrol'
 'nominal__Transmission_Automatic' 'nominal__Transmission_Manual'
 'nominal__City_Ahmedabad' 'nominal__City_Bengaluru'
 'nominal__City_Chandigarh' 'nominal__City_Delhi'
 'nominal__City_Hyderabad' 'nominal__City_Jaipur' 'nominal__City_Kochi'
 'nominal__City_Lucknow' 'nominal__City_Mumbai' 'nominal__City_Pune'
 'nominal__Seller_Type_Certified Dealer' 'nominal__Seller_Type_Dealer'
 'nominal__Seller_Type_Indiv

## Step 20: Convert Processed Arrays into DataFrames
Processed arrays are converted into readable Pandas DataFrames.

In [ ]:
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)
X_train_processed_df.head()

,numerical__Year,numerical__Mileage_Km,numerical__Engine_CC,numerical__Power_BHP,numerical__Previous_Owners,numerical__Accidents_Reported,numerical__Service_Score,ordinal__Condition,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,nominal__Brand_Mahindra,nominal__Brand_Maruti,nominal__Brand_Renault,nominal__Brand_Skoda,nominal__Brand_Tata,nominal__Brand_Toyota,nominal__Brand_Volkswagen,nominal__Fuel_Type_CNG,nominal__Fuel_Type_Diesel,nominal__Fuel_Type_Electric,nominal__Fuel_Type_Petrol,nominal__Transmission_Automatic,nominal__Transmission_Manual,nominal__City_Ahmedabad,nominal__City_Bengaluru,nominal__City_Chandigarh,nominal__City_Delhi,nominal__City_Hyderabad,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## Step 21: Add the Target Variable Back
The target is added back after feature preprocessing.

In [ ]:
train_processed = X_train_processed_df.copy()
test_processed = X_test_processed_df.copy()
train_processed[target] = y_train.values
test_processed[target] = y_test.values
display(train_processed.head())

,numerical__Year,numerical__Mileage_Km,numerical__Engine_CC,numerical__Power_BHP,numerical__Previous_Owners,numerical__Accidents_Reported,numerical__Service_Score,ordinal__Condition,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,nominal__Brand_Mahindra,nominal__Brand_Maruti,nominal__Brand_Renault,nominal__Brand_Skoda,nominal__Brand_Tata,nominal__Brand_Toyota,nominal__Brand_Volkswagen,nominal__Fuel_Type_CNG,nominal__Fuel_Type_Diesel,nominal__Fuel_Type_Electric,nominal__Fuel_Type_Petrol,nominal__Transmission_Automatic,nominal__Transmission_Manual,nominal__City_Ahmedabad,nominal__City_Bengaluru,nominal__City_Chandigarh,nominal__City_Delhi,nominal__City_Hyderabad,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,Resale_Price_Lakh
0,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
1,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
2,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
3,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
4,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69


# Dataset Verification
## Step 22: Verify the Processed Datasets
Check shapes, missing values, duplicates, and successful preprocessing.

In [ ]:
print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)
print("Train missing:", train_processed.isnull().sum().sum())
print("Test missing:", test_processed.isnull().sum().sum())
print("Train duplicates:", train_processed.duplicated().sum())
print("Test duplicates:", test_processed.duplicated().sum())

Train shape: (256, 38)
Test shape: (64, 38)
Train missing: 0
Test missing: 0
Train duplicates: 0
Test duplicates: 0


## Step 23: Verify Numerical Feature Scaling
Training numerical features should be centered approximately around zero after standardization.

In [ ]:
numerical_processed_columns = ["numerical__Year","numerical__Mileage_Km","numerical__Engine_CC","numerical__Power_BHP","numerical__Previous_Owners","numerical__Accidents_Reported","numerical__Service_Score"]
train_processed[numerical_processed_columns].describe()

,numerical__Year,numerical__Mileage_Km,numerical__Engine_CC,numerical__Power_BHP,numerical__Previous_Owners,numerical__Accidents_Reported,numerical__Service_Score
count,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02,2.560000e+02,256.0,2.560000e+02
mean,-3.816392e-17,1.040834e-17,3.469447e-18,-7.077672e-16,-4.163336e-17,0.0,-6.938894e-18
std,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00,1.001959e+00,0.0,1.001959e+00
min,-1.698227e+00,-2.046919e+00,-1.630394e+00,-2.597881e+00,-7.557515e-01,0.0,-1.738645e+00
25%,-7.893502e-01,-7.700529e-01,-7.114669e-01,-6.424083e-01,-7.557515e-01,0.0,-8.755947e-01
50%,1.195268e-01,-1.038622e-02,-5.604318e-02,1.216614e-02,-7.557515e-01,0.0,1.078813e-01
75%,7.254448e-01,6.888949e-01,6.927743e-01,6.612399e-01,4.844561e-01,0.0,7.501513e-01
max,1.634322e+00,2.877317e+00,2.799136e+00,2.616712e+00,2.344768e+00,0.0,1.713556e+00


# Save the Preprocessed Dataset
## Step 24: Create the Complete Processed Dataset
The complete dataset is transformed using the preprocessor already fitted on training data. No new fitting is performed.

In [ ]:
X_processed = preprocessor.transform(X)
processed_df = pd.DataFrame(X_processed, columns=feature_names)
processed_df[target] = y.values
processed_df.head()

,numerical__Year,numerical__Mileage_Km,numerical__Engine_CC,numerical__Power_BHP,numerical__Previous_Owners,numerical__Accidents_Reported,numerical__Service_Score,ordinal__Condition,nominal__Brand_Honda,nominal__Brand_Hyundai,nominal__Brand_Kia,nominal__Brand_Mahindra,nominal__Brand_Maruti,nominal__Brand_Renault,nominal__Brand_Skoda,nominal__Brand_Tata,nominal__Brand_Toyota,nominal__Brand_Volkswagen,nominal__Fuel_Type_CNG,nominal__Fuel_Type_Diesel,nominal__Fuel_Type_Electric,nominal__Fuel_Type_Petrol,nominal__Transmission_Automatic,nominal__Transmission_Manual,nominal__City_Ahmedabad,nominal__City_Bengaluru,nominal__City_Chandigarh,nominal__City_Delhi,nominal__City_Hyderabad,nominal__City_Jaipur,nominal__City_Kochi,nominal__City_Lucknow,nominal__City_Mumbai,nominal__City_Pune,nominal__Seller_Type_Certified Dealer,nominal__Seller_Type_Dealer,nominal__Seller_Type_Individual,Resale_Price_Lakh
0,0.422486,-0.102145,-0.402934,-0.669911,-0.755752,0.0,-0.373821,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,6.38
1,0.119527,0.439757,-0.956625,-0.113562,-0.755752,0.0,0.830435,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,4.83
2,0.422486,-0.838755,0.250822,1.124864,0.484456,0.0,1.071286,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,7.30
3,-0.183432,-0.069952,1.636162,-0.041269,1.724664,0.0,-0.855524,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.82
4,-1.092309,0.788730,0.720014,1.756650,0.484456,0.0,0.589584,-1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.93


## Step 25: Save the Processed Files
Save the complete processed dataset, train set, test set, and outlier report.

In [ ]:
processed_df.to_csv("Used_Car_Preprocessed_Dataset.csv", index=False)
train_processed.to_csv("Used_Car_Preprocessed_Train.csv", index=False)
test_processed.to_csv("Used_Car_Preprocessed_Test.csv", index=False)
outlier_report.to_csv("Used_Car_Training_Outlier_Report.csv", index=False)
print("All files saved successfully!")

All files saved successfully!


# Final Results
The workflow completed dataset inspection, train/test splitting, IQR outlier handling, ordinal and nominal encoding, numerical scaling, verification, and saving. All transformations were fitted only on training data to prevent data leakage.

# Conclusion
A complete and leakage-safe preprocessing workflow was performed. The final dataset is suitable for machine learning models that predict used car resale prices.